# Checking the shared evaluation script

This notebook downloads the repo fresh,
so it tests exactly what our team will get. No GPU needed

What each cell proves:
1. the repo downloads and the packages install
2. all automated tests pass
3. the "dumb" systems give the expected numbers with the NEW official scorer
4. the evaluation script scores a real system (Assignment 1 Baseline B) end to end
5. the comparison test works

In [6]:
# Cell 1: fresh copy of the team repo + packages
%cd /content
!rm -rf SenseShift
!git clone -q https://github.com/abiha25/SenseShift-Semantic-change-across-multiple-time-periods.git SenseShift
%cd /content/SenseShift
!pip install -q bcubed nltk pyyaml
!git log --oneline -3

/content
/content/SenseShift
ac15d65 (HEAD -> main, origin/main, origin/HEAD) Add files via upload
dd83d55 Add files via upload
825a17c Add files via upload


**Cell 2 - tests.** Expected output line is: `31 passed`.

In [7]:
# Cell 2: automated tests
!python -m pytest tests -q 2>&1 | tail -3


-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
31 passed, 1 warning in 14.49s


**Cell 3 - sanity check.** Expected: one_cluster `B3 P 0.646  B3 R 0.981  B3 F1 0.756  rho 0.000`;
gold_labels all `1.000`; Subtask 2 `all_1 = 0.236`, `all_0 = 0.361`.

(Before the organisers' update of 8 Nov 2026, all_1 was 0.472 and P/R were swapped.)

In [8]:
# Cell 3: sanity check with the new official scorer
!python src/sanity_check.py

Subtask 1 (SV):
  system                  B3 P    B3 R   B3 F1   1b rho
  one_cluster            0.646   0.981   0.756    0.000
  each_own_cluster       1.000   0.027   0.052   -0.174
  random_5_clusters      0.656   0.219   0.315   -0.077
  gold_labels            1.000   1.000   1.000    1.000

Subtask 2 (SV):
  all_1                macro F1 = 0.236
  all_0                macro F1 = 0.361
  random               macro F1 = 0.439
  gold_labels          per word = {'dumpa': 1.0, 'exportera': 1.0, 'fru': 1.0, 'fröken': 1.0, 'förort': 1.0, 'herre': 1.0, 'kvinna': 1.0, 'motiv': 1.0, 'skär': 1.0, 'suga': 1.0}

Saved to results/sanity/summary.json


**Cell 4 - score a real system.** Assignment 1 Baseline B (XL-LEXEME). Expected: 1a `0.760`, 1b `0.492`
(same as Assignment 1, because these metrics did not change), and Subtask 2 `0.282` under the new macro F1
(the old metric, shown as "positive-class F1 (A1 metric)", is still `0.468`).

In [9]:
# Cell 4: full evaluation report of a real system
!python -m src.evaluate score --name A1_baselineB_xllex \
    --s1 results/pretrained/SV_subtask1_xllex_combined.jsonl \
    --s2 results/pretrained/SV_subtask2_xllex_cos_rankthr.jsonl \
    --out /content/eval_check/B --subgroups

Evaluation: A1_baselineB_xllex   (SV, 10 words)
Format check subtask1: OK
Format check subtask2: OK

OFFICIAL METRICS (95% CI over words)
  1a  BCubed F1        0.760  [0.657, 0.852]   (P 0.689, R 0.900; one cluster = 0.756)
  1b  Spearman rho     0.492  [0.176, 0.748]   (60 pairs)
  2   macro F1         0.282  [0.199, 0.371]

SECONDARY METRICS
  1b  Pearson r 0.450 | rho within-era 0.529 (56 pairs) | rho vs cross-era flag 0.142
      shuffle control: 0.384 +- 0.037 (200 draws), p = 0.000
  2   accuracy 0.381 | balanced accuracy 0.510 | positive-class F1 (A1 metric) 0.468   (means over words)
      pooled over usages: accuracy 0.373, balanced accuracy 0.504, macro F1 0.307

PER WORD
  word         1a F1  1clust  #pred  #gold   2 mF1  gold+  pred+
  dumpa        0.849   0.849      1      6   0.282   0.31   0.95
  exportera    0.946   0.946      1      3   0.544   0.95   0.95
  fru          0.896   0.896      1      5   0.510   0.89   0.95
  fröken       0.819   0.819      1      4   0.0

**Cell 5 - comparing two systems** (Assignment 1 Baseline B vs Baseline A, Subtask 1a).
Expected: difference about `+0.063`, p = `0.5`, verdict *not significant* - with 10 words, a gain of
0.06 is not enough evidence.

In [10]:
# Cell 5: is B really better than A?
!python -m src.evaluate score --name A1_baselineA --s1 results/classical/SV_subtask1_combined.jsonl \
    --s2 results/classical/SV_subtask2_LR_rank_thr_0.5.jsonl --out /content/eval_check/A > /dev/null
!python -m src.evaluate compare /content/eval_check/B/scores.json /content/eval_check/A/scores.json --metric 1a

{
  "a": "A1_baselineB_xllex",
  "b": "A1_baselineA",
  "metric_a": "1a",
  "metric_b": "1a",
  "n_words": 10,
  "difference_a_minus_b": 0.06318783430223415,
  "ci": {
    "low": -0.026783462749603266,
    "high": 0.19695749929901307,
    "level": 0.95
  },
  "p_value": 0.5,
  "test": "exact paired sign-flip permutation test",
  "verdict": "not significant: report as a difference, not an improvement"
}


Done.